# 03 - Is the difference significant? (Diebold-Mariano, Clark-West, Holm)

A small positive $R^2_{OOS}$ gain could be luck. We need a test of
$$H_0:\ E[e_{base}^2-e_{alt}^2]\le 0\quad\text{vs}\quad H_1:\ E[e_{base}^2-e_{alt}^2]> 0 .$$

## Diebold-Mariano (1995)
Let $d_t=e_{base,t}^2-e_{alt,t}^2$ (loss differential). Test whether its mean is zero with
$t=\bar d/\sqrt{\widehat{\mathrm{Var}}(\bar d)}$.

**HAC / Newey-West variance.** Our target is a 5-day rolling std, so consecutive targets share 4 of 5 returns: forecast errors, hence $d_t$, are
*autocorrelated*. The naive variance $\sigma^2/n$ would be too small and the test would reject too often. Newey-West estimates
$\widehat{\mathrm{Var}}(\bar d)=\frac1n\big(\hat\gamma_0+2\sum_{k=1}^{L}(1-\tfrac{k}{L+1})\hat\gamma_k\big)$ with $L=5$ lags, robust to this serial correlation.

## Clark-West (2007) - why DM is not enough here
Our two models are **nested** (the attention model contains the baseline). If $H_0$ is true the extra coefficients are zero, but we
*estimate* them anyway, adding pure estimation noise, so the bigger model is biased to look *worse* out of sample. DM is then too
conservative (rarely rejects). Clark-West corrects the loss differential:
$$\tilde d_t=e_{base,t}^2-\big(e_{alt,t}^2-(\hat y^{base}_t-\hat y^{alt}_t)^2\big)$$
and tests its mean in the same way. It is the appropriate test for nested *linear* models, and it is more powerful.

## Holm correction
We test 6 pairs. At the 5% level, testing 6 true nulls gives about a 26% chance of at least one false positive
($1-0.95^6$). Holm's step-down procedure controls the family-wise error rate: sort p-values, compare the $k$-th smallest with $\alpha/(m-k+1)$.

In [ ]:
import os, sys, warnings
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from src import model_pipeline as mp
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": .3})
START, END = "2016-01-01", "2026-10-31"

In [ ]:
df = pd.read_csv("results/results.csv")
df[["ticker", "page", "r2_oos_gain", "dm_stat", "p_dm", "p_dm_holm", "cw_stat", "p_cw", "p_cw_holm"]].round(4)

**Reading.**
- Under **DM**, nothing is close to significant (best raw p=0.14).
- Under the more powerful **Clark-West**, XLK/Apple reaches raw p=0.019 and SPY/Fed p=0.099 - the only hints of a signal.
  But with six tests a raw p of 0.019 is not impressive: after **Holm** the best adjusted p is 0.11. **We do not reject the null on any pair at 5%.**
- Honest statement: *a weak hint on XLK/Apple (tech ETF, Apple article) that does not survive multiple-testing correction.*
  Choosing XLK/Apple *after* seeing this would be p-hacking; it would need a fresh out-of-sample period to confirm.

## Cumulative loss differential

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
for t, p in mp.PAIRS:
    _, dl = mp.evaluate(t, p, START, END)
    dl.cumsum().plot(ax=ax, lw=1, label=f"{t}/{p}")
ax.axhline(0, color="k", lw=.8); ax.legend(fontsize=7)
ax.set_title("Cumulative (baseline error^2 - attention error^2): rising = attention helping")
plt.show()

A genuine predictive signal would show a steadily rising line. Here lines wander around zero (XLK/Apple drifts up slowly, QQQ/Nvidia drifts down).

## Sanity check by simulation: what do the tests do when there is *no* signal?
We simulate a persistent AR(1) series and an *irrelevant* regressor, run the same walk-forward + DM/CW, and look
at the rejection rate at 5% (a valid test should give $\le 5\%$).

In [ ]:
rng = np.random.default_rng(0)
def one_sim(n=1500):
    e = rng.normal(size=n); y = np.zeros(n)
    for t in range(1, n): y[t] = 0.9 * y[t-1] + e[t]
    z = rng.normal(size=n)                       # pure noise "attention"
    X = pd.DataFrame({"y1": np.r_[0, y[:-1]], "z1": np.r_[0, z[:-1]], "z2": np.r_[0, 0, z[:-2]]})
    ys = pd.Series(y)
    pb = mp.walk_forward(X, ys, ["y1"], min_train=500, refit=21)
    pa = mp.walk_forward(X, ys, ["y1", "z1", "z2"], min_train=500, refit=21)
    r, _ = mp.compare(ys, pb, pa)
    return r["p_dm"], r["p_cw"]
ps = np.array([one_sim() for _ in range(150)])
rej = (ps < .05).mean(0)
print(f"rejection rate at 5 pct level -> DM: {rej[0]:.3f}   Clark-West: {rej[1]:.3f}")

DM rejects far less than 5% (too conservative for nested models); Clark-West is closer to the nominal size. That is why we report both, and why the
conclusion "not significant after correction" is not an artifact of a timid test.